# VIX Deep Learning Pipeline V2 — Extensions Avancées
## Calibration · Stacking · Mamba · GNN · Conformal Prediction · Adversarial Validation · Stress Testing · Ensemble Asymétrique


## 0. Installation et imports

In [38]:
import sys
!{sys.executable} -m pip install -q arch pykalman hmmlearn shap xlsxwriter mapie
!{sys.executable} -m pip install -q torch-geometric 2>/dev/null || True

# Mamba nécessite CUDA + compilation C++ — désactivé par défaut
MAMBA_AVAILABLE = False

import os, time, random, warnings, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.mixture import GaussianMixture
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                              precision_score, recall_score, brier_score_loss)
from sklearn.calibration import calibration_curve   # ← dans sklearn.calibration
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier

from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from imblearn.combine import SMOTETomek

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

from arch import arch_model
from pykalman import KalmanFilter
from hmmlearn import hmm as hmmlib

try:
    from mapie.classification import MapieClassifier
    MAPIE_AVAILABLE = True
except ImportError:
    MAPIE_AVAILABLE = False
    print("[INFO] MAPIE non disponible")

try:
    from torch_geometric.nn import GATConv, GCNConv
    from torch_geometric.data import Data
    GNN_AVAILABLE = True
except ImportError:
    GNN_AVAILABLE = False
    print("[INFO] PyTorch Geometric non disponible — GNN skipped")

# ── Reproductibilité ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device} | GNN : {GNN_AVAILABLE} | Mamba : {MAMBA_AVAILABLE}")


[INFO] MAPIE non disponible
Device : cpu | GNN : True | Mamba : False


## 1. Configuration

In [39]:
CONFIG = {
    'seed': 42, 'start_date': '2012-01-01',
    'lookback': 21, 'horizons': [1, 3, 5, 7],
    'flat_thr': 0.003,
    'top_n_shap': 40, 'top_n_final': 30,
    'batch_size': 64, 'epochs': 60, 'lr': 3e-4,
    'weight_decay': 1e-4, 'dropout': 0.3,
    'class_quantiles': [0.25, 0.75],
    'class_labels': ['DOWN_FORT','DOWN_FAIBLE','UP_FAIBLE','UP_FORT'],
    'temperature_lr': 0.01,
    'temperature_epochs': 100,
    'conformal_alpha': 0.10,
    'adv_val_threshold': 0.70,
}

TARGET_COL = 'VIX_Amplitude_Class'

YF_TICKERS = """
^GSPC ^IXIC ^VIX ^VXN ^OVX ^GVZ ^EVZ ^VVIX
^FTSE ^N225 ^HSI ^GDAXI ^STOXX50E
SPY QQQ TLT GLD USO HYG LQD
AAPL AMZN MSFT NVDA INTC QCOM XOM WMT MCD SBUX
MS COF BLK SCHW CLX CPB LMT NOC GD HON
CCI PSA EQIX NEE TXN PAYX LUV CMCSA
XLK XLF XLE XLV XLU XLB XLI XLY
""".split()

FRED_SERIES = {
    'NFCI':   'NFCI',
    'STLFSI': 'STLFSI4',
    'T10Y2Y': 'T10Y2Y',
    'EFFR':   'EFFR',
}

CRISIS_PERIODS = {
    'GFC_2008':      ('2008-09-01', '2009-03-31'),
    'Euro_2011':     ('2011-07-01', '2012-01-31'),
    'COVID_2020':    ('2020-02-20', '2020-05-31'),
    'Fed_Hike_2022': ('2022-01-01', '2022-12-31'),
    'SVB_2023':      ('2023-03-01', '2023-05-31'),
}

REGIME_THRESHOLDS = {'calm': 18.0, 'stress': 25.0}

print("Configuration V2 chargée.")


Configuration V2 chargée.


## 2. Chargement des données

**Filtre par colonne** : on retire les séries avec moins de 90% de données disponibles,
jamais par ligne — évite de perdre des années entières à cause d'un ticker récent.


In [40]:
t0 = time.time()

def load_data(start=CONFIG['start_date']):
    """Télécharge Yahoo Finance + FRED et retourne un DataFrame propre."""
    # Yahoo Finance
    raw = yf.download(YF_TICKERS, start=start, auto_adjust=True, progress=False)['Close']
    raw.columns = [c.replace('^','IDX_').replace('-','_') for c in raw.columns]

    # Filtre couverture colonnes (≥90%), jamais de filtre par ligne
    coverage = raw.notna().mean()
    raw = raw.loc[:, coverage >= 0.90]
    raw = raw.ffill().dropna(how='all')

    # FRED
    fred_frames = []
    for name, sid in FRED_SERIES.items():
        try:
            s = web.DataReader(sid, 'fred', start).squeeze()
            s.name = f'FRED_{name}'
            fred_frames.append(s)
        except Exception as e:
            print(f"  [WARN] FRED {sid}: {e}")

    if fred_frames:
        fred_df = pd.concat(fred_frames, axis=1).reindex(raw.index, method='ffill')
        raw = pd.concat([raw, fred_df], axis=1)

    print(f"  Dataset : {raw.shape[0]} jours × {raw.shape[1]} séries ({time.time()-t0:.1f}s)")
    return raw

df_raw = load_data()


  Dataset : 3784 jours × 59 séries (6.9s)


## 3. Feature Engineering avancé

Regroupe les features de base (vol-of-vol, momentum, RSI, MACD, Bollinger, drawdown)
utilisées en V1. Calculées ici pour être réutilisables dans le pipeline V2.


In [41]:
def build_advanced_features(df_raw, vix_series, spx_series, train_end_idx):
    """Features d'amplitude et momentum sur VIX et SPX."""
    feats = pd.DataFrame(index=df_raw.index)
    vix = vix_series.ffill()
    spx = spx_series.ffill()
    vix_ret = np.log(vix / vix.shift(1))
    spx_ret = np.log(spx / spx.shift(1))

    feats['vix_vol_of_vol_5d']  = vix_ret.rolling(5,  min_periods=3).std()
    feats['vix_vol_of_vol_10d'] = vix_ret.rolling(10, min_periods=5).std()
    feats['vix_momentum_2d']    = vix.pct_change(2)
    feats['vix_momentum_3d']    = vix.pct_change(3)
    feats['vix_acceleration']   = vix_ret - vix_ret.shift(3)
    feats['vix_erratic_ratio']  = (vix_ret.abs().rolling(5,min_periods=3).max() /
                                    vix_ret.abs().rolling(5,min_periods=3).mean().replace(0,np.nan))
    feats['vix_vol_ratio_5_60'] = (vix_ret.rolling(5,min_periods=3).std() /
                                    vix_ret.rolling(60,min_periods=30).std().replace(0,np.nan))
    feats['spx_vol_5d']         = spx_ret.rolling(5, min_periods=3).std()
    feats['spx_momentum_3d']    = spx.pct_change(3)
    feats['vix_spx_corr_30d']   = vix_ret.rolling(30, min_periods=15).corr(spx_ret)

    for w in [5, 10, 20]:
        ma = vix.rolling(w, min_periods=w//2).mean()
        feats[f'vix_vs_ma{w}']     = (vix - ma) / ma.replace(0,np.nan)
        feats[f'vix_zscore_{w}d']  = (vix - ma) / vix.rolling(w,min_periods=w//2).std().replace(0,np.nan)

    for h in [1,2,3,5,10,20]:
        feats[f'vix_ret_{h}d'] = vix.pct_change(h)
        feats[f'spx_ret_{h}d'] = spx.pct_change(h)

    def rsi(s, n=14):
        d = s.diff(); g = d.clip(lower=0).rolling(n).mean(); l = (-d.clip(upper=0)).rolling(n).mean()
        return 100 - 100/(1+g/l.replace(0,np.nan))
    feats['vix_rsi_14'] = rsi(vix); feats['spx_rsi_14'] = rsi(spx)

    for w in [10,20]:
        ma = vix.rolling(w).mean(); std = vix.rolling(w).std()
        feats[f'boll_width_{w}d'] = (2*std) / ma.replace(0,np.nan)

    ema12 = vix.ewm(span=12).mean(); ema26 = vix.ewm(span=26).mean()
    macd  = ema12 - ema26; sig = macd.ewm(span=9).mean()
    feats['vix_macd'] = macd; feats['vix_macd_signal'] = sig; feats['vix_macd_hist'] = macd - sig

    feats['vix_roll_skew_20d'] = vix_ret.rolling(20).skew()
    feats['vix_roll_kurt_20d'] = vix_ret.rolling(20).kurt()

    roll_high = spx.rolling(252, min_periods=126).max()
    feats['spx_drawdown_252d'] = (spx - roll_high) / roll_high.replace(0,np.nan)

    return feats.replace([np.inf,-np.inf], np.nan)


## 4. Features Time-Series : EGARCH, Kalman, HMM, Heston, VRP, Hawkes

**Règle absolue** : tous les fits (EGARCH, Kalman EM, HMM Baum-Welch, HAR-RV)
sont effectués sur `df_raw.iloc[:train_end_idx]` uniquement.
Le modèle est ensuite appliqué à tout le dataset (pas de re-fit sur le test).


In [42]:
def build_ts_features(df_raw, train_end_idx):
    """Construit toutes les features time-series sans leakage."""
    feats = pd.DataFrame(index=df_raw.index)
    t0 = time.time()

    vix_col  = [c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX'))
                 and 'VXN' not in c and 'VVIX' not in c][0]
    spx_cols = [c for c in df_raw.columns if 'GSPC' in c or (c=='SPY')]
    spx_col  = spx_cols[0] if spx_cols else None

    vix     = df_raw[vix_col].ffill()
    vix_ret = np.log(vix / vix.shift(1)).fillna(0)

    # ── EGARCH SPX ────────────────────────────────────────────────────────────
    if spx_col:
        spx      = df_raw[spx_col].ffill()
        spx_ret  = np.log(spx / spx.shift(1)).fillna(0)
        spx_tr   = spx_ret.iloc[:train_end_idx] * 100
        try:
            am  = arch_model(spx_tr, vol='EGARCH', p=1, q=1, dist='skewt', rescale=False)
            res = am.fit(disp='off', show_warning=False)
            fc  = res.forecast(start=0, reindex=True)
            condvar = (fc.variance.iloc[:,0] / 10000).reindex(df_raw.index, method='ffill')
            feats['egarch_condvar'] = condvar.replace([np.inf,-np.inf], np.nan)
            feats['egarch_delta']   = condvar.diff()
            print(f"  EGARCH OK ({time.time()-t0:.1f}s)")
        except Exception as e:
            print(f"  [WARN] EGARCH: {e}")

    # ── Kalman ────────────────────────────────────────────────────────────────
    try:
        vix_clean = vix.ffill().fillna(vix.mean())
        vix_arr   = np.where(np.isfinite(vix_clean.values), vix_clean.values,
                             np.nanmean(vix_clean.values)).reshape(-1,1)
        kf = KalmanFilter(transition_matrices=[[1]], observation_matrices=[[1]],
                          initial_state_mean=[vix_clean.iloc[0]],
                          initial_state_covariance=[[1.]],
                          em_vars=['transition_covariance','observation_covariance'])
        kf = kf.em(vix_arr[:train_end_idx], n_iter=20)
        sm, _ = kf.filter(vix_arr)
        ss, _ = kf.smooth(vix_arr)
        kalman_f = pd.Series(sm[:,0], index=df_raw.index)
        kalman_s = pd.Series(ss[:,0], index=df_raw.index)
        # Shift de 1j anti-leakage : innovation_t utilise VIX_t qui est au dénominateur de la cible
        feats['kalman_residual']   = (vix_clean - kalman_f).shift(1).replace([np.inf,-np.inf], np.nan)
        feats['kalman_innovation'] = (vix_clean - kalman_s.shift(1)).shift(1).replace([np.inf,-np.inf], np.nan)
        print(f"  Kalman OK ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] Kalman: {e}")

    # ── HMM ───────────────────────────────────────────────────────────────────
    try:
        rv5   = vix_ret.pow(2).rolling(5, min_periods=3).mean()
        vix_n = (vix - vix.iloc[:train_end_idx].mean()) / vix.iloc[:train_end_idx].std()
        X_hmm = pd.DataFrame({'ret':vix_ret,'vol5':np.sqrt(rv5),'level':vix_n}).dropna()
        X_tr  = X_hmm.iloc[:train_end_idx].values
        mh    = hmmlib.GaussianHMM(n_components=2, covariance_type='full',
                                    n_iter=200, random_state=SEED)
        mh.fit(X_tr)
        states_tr  = mh.predict(X_tr)
        state_vols = [rv5.reindex(X_hmm.index[:train_end_idx]).values[states_tr==s].mean() for s in range(2)]
        stress_st  = int(np.argmax(state_vols))
        proba_full = mh.predict_proba(X_hmm.values)
        feats['hmm_p_stress'] = pd.Series(proba_full[:,stress_st], index=X_hmm.index).reindex(df_raw.index)
        print(f"  HMM OK ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] HMM: {e}")

    # ── Heston proxies ────────────────────────────────────────────────────────
    try:
        v0    = (vix / 100).pow(2)
        theta = vix_ret.pow(2).rolling(60, min_periods=30).mean()
        vvix_cols = [c for c in df_raw.columns if 'VVIX' in c]
        xi    = (df_raw[vvix_cols[0]].ffill() / 100) if vvix_cols else vix_ret.rolling(20).std()
        xi    = xi.reindex(df_raw.index, method='ffill')

        if spx_col:
            spx = df_raw[spx_col].ffill()
            spx_r = np.log(spx/spx.shift(1)).fillna(0)
            rho = vix_ret.rolling(30, min_periods=15).corr(spx_r)
        else:
            rho = pd.Series(-0.7, index=df_raw.index)

        def rolling_kappa(s, w=252):
            k = pd.Series(np.nan, index=s.index)
            sf = s.ffill().fillna(0)
            for i in range(w, len(sf)):
                y_ = sf.iloc[i-w+1:i+1].values; x_ = sf.iloc[i-w:i].values
                try:
                    b = np.corrcoef(x_,y_)[0,1]
                    if np.isfinite(b) and 0 < abs(b) < 0.9999:
                        hl = -np.log(2)/np.log(abs(b))
                        if np.isfinite(hl) and hl > 0: k.iloc[i] = np.log(2)/hl
                except: pass
            return k.replace([np.inf,-np.inf], np.nan)

        kappa = rolling_kappa(vix)
        feats['heston_v0']    = v0
        feats['heston_theta'] = theta
        feats['heston_xi']    = xi
        feats['heston_rho']   = rho
        feats['heston_kappa'] = kappa
        feats['heston_feller']= (2*kappa*theta) / xi.pow(2).replace(0,np.nan)
        feats['heston_v0_minus_theta'] = v0 - theta

        for h in [1,3,5,7]:
            ev = theta + (v0 - theta) * np.exp(-kappa * h)
            feats[f'heston_ev_h{h}']     = ev
            feats[f'heston_spread_h{h}'] = v0 - ev
            feats[f'heston_vol_h{h}']    = np.sqrt(ev.clip(0)) * 100
        print(f"  Heston OK ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] Heston: {e}")

    # ── VRP via HAR-RV ────────────────────────────────────────────────────────
    try:
        import statsmodels.api as sm
        rv1d  = vix_ret.pow(2).replace([np.inf,-np.inf], np.nan)
        rv5d  = rv1d.rolling(5,  min_periods=3).mean()
        rv22d = rv1d.rolling(22, min_periods=10).mean()
        rv_tgt = rv1d.shift(-22).rolling(22, min_periods=11).mean()
        hdf   = pd.DataFrame({'rv1':rv1d,'rv5':rv5d,'rv22':rv22d,'y':rv_tgt}).dropna()
        hdf   = hdf.replace([np.inf,-np.inf], np.nan).dropna()
        tr_m  = hdf.index[:train_end_idx]
        if len(tr_m) > 10:
            Xh  = sm.add_constant(hdf.loc[tr_m,['rv1','rv5','rv22']])
            Xh  = Xh.replace([np.inf,-np.inf], np.nan).dropna()
            yh  = hdf.loc[Xh.index,'y']
            hm  = sm.OLS(yh, Xh).fit()
            Xf  = sm.add_constant(hdf[['rv1','rv5','rv22']]).replace([np.inf,-np.inf],0).fillna(0)
            rv_pred = hm.predict(Xf).reindex(df_raw.index).fillna(0)
        else:
            rv_pred = pd.Series(0., index=df_raw.index)
        feats['VRP'] = (vix/100).pow(2) - rv_pred
        mu_vrp = feats['VRP'].iloc[:train_end_idx].mean()
        sd_vrp = feats['VRP'].iloc[:train_end_idx].std()
        feats['VRP_zscore'] = (feats['VRP'] - mu_vrp) / sd_vrp
        print(f"  VRP OK ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] VRP: {e}")

    # ── Jump Intensity ────────────────────────────────────────────────────────
    sig60 = vix_ret.rolling(60, min_periods=30).std()
    feats['jump_intensity_20d'] = (vix_ret.abs() > 3*sig60).astype(float).rolling(20,min_periods=10).mean()
    feats['jump_intensity_60d'] = (vix_ret.abs() > 3*sig60).astype(float).rolling(60,min_periods=30).mean()

    # ── Hawkes ────────────────────────────────────────────────────────────────
    try:
        sig_hw  = vix_ret.rolling(30, min_periods=15).std()
        jt      = vix_ret.index[vix_ret.abs() > 2*sig_hw]
        hawkes  = pd.Series(0., index=vix_ret.index)
        for idx_i, t in enumerate(vix_ret.index):
            past = jt[jt < t]
            if len(past):
                days = np.array([(t-tj).days for tj in past])
                hawkes.iloc[idx_i] = 0.3 + 0.3*np.sum(np.exp(-0.1*days))
            else:
                hawkes.iloc[idx_i] = 0.3
        mu_hw = hawkes.iloc[:train_end_idx].mean()
        sd_hw = hawkes.iloc[:train_end_idx].std()
        feats['hawkes_intensity'] = hawkes
        feats['hawkes_zscore']    = (hawkes - mu_hw) / (sd_hw + 1e-8)
        print(f"  Hawkes OK ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  [WARN] Hawkes: {e}")

    return feats.replace([np.inf,-np.inf], np.nan)


## 5. Cible amplitude 4 classes (quantiles conditionnels par régime)

In [43]:
def build_amplitude_target(vix_series, horizon, train_end_idx):
    vix = vix_series.ffill()
    vix_tr = vix.iloc[:train_end_idx]
    calm_thr   = vix_tr.quantile(0.33)
    stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr]    = 'CALM'
    regime[vix >= stress_thr] = 'STRESS'

    ret = (vix.shift(-horizon) / vix) - 1
    flat_mask = ret.abs() < CONFIG['flat_thr']
    ret    = ret.loc[~flat_mask].dropna()
    regime = regime.reindex(ret.index)

    ret_train = ret.iloc[:train_end_idx]
    reg_train = regime.iloc[:train_end_idx]
    thresholds = {}
    for reg in ['CALM','NORMAL','STRESS']:
        sub = ret_train[reg_train == reg]
        q25 = sub.quantile(0.25) if len(sub)>=20 else ret_train.quantile(0.25)
        q75 = sub.quantile(0.75) if len(sub)>=20 else ret_train.quantile(0.75)
        thresholds[reg] = (q25, q75)
        print(f"  [{reg}] q25={q25:.3%} q75={q75:.3%} n={len(sub)}")
    thresholds['GLOBAL'] = (ret_train.quantile(0.25), ret_train.quantile(0.75))

    def classify(r, reg):
        q25, q75 = thresholds.get(reg, (0,0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3

    target = pd.Series([classify(r, regime[i]) for i,r in ret.items()],
                        index=ret.index, name=TARGET_COL)
    print(f"  [Target h={horizon}j] {len(target)} obs | {target.value_counts().to_dict()}")
    return target, regime, ret, thresholds


## 6. Dataset PyTorch, Interactions SHAP

In [44]:
class VIXAmplitudeDataset(Dataset):
    def __init__(self, data, feature_cols, target_col=TARGET_COL,
                 lookback=CONFIG['lookback'], scaler=None):
        self.lookback = lookback
        X = data[feature_cols].copy().fillna(0).replace([np.inf,-np.inf],0).astype(float)
        y = data[target_col].values.astype(int)
        self.scaler = scaler if scaler is not None else RobustScaler()
        self.X = (self.scaler.fit_transform(X) if scaler is None
                  else self.scaler.transform(X))
        self.y = y
        self.n_features = self.X.shape[1]

    def __len__(self):  return max(0, len(self.X) - self.lookback)
    def __getitem__(self, i):
        return (torch.tensor(self.X[i:i+self.lookback], dtype=torch.float32),
                torch.tensor(self.y[i+self.lookback], dtype=torch.long))


def generate_interactions(df, base_features, top_n=20, rolling_w=20, eps=1e-8):
    feats = [f for f in base_features[:top_n] if f in df.columns]
    cols  = {}
    for i in range(len(feats)):
        for j in range(i+1, len(feats)):
            fi,fj = feats[i],feats[j]
            si,sj = df[fi],df[fj]
            sd = sj.where(sj.abs()>=eps, np.nan)
            cols[f'{fi}__div__{fj}']     = si/sd
            cols[f'{fi}__minus__{fj}']   = si-sj
            cols[f'{fi}__prod__{fj}']    = si*sj
            d   = si-sj; rs = d.rolling(rolling_w,min_periods=rolling_w//2).std()
            cols[f'{fi}__zrel__{fj}']    = d/rs.replace(0,np.nan)
            mai = si.rolling(rolling_w,min_periods=rolling_w//2).mean()
            maj = sj.rolling(rolling_w,min_periods=rolling_w//2).mean()
            cols[f'{fi}__macross__{fj}'] = mai/maj.where(maj.abs()>=eps,np.nan)
            cols[f'{fi}__ret5x__{fj}']   = si.pct_change(5)*sj
    idf = pd.DataFrame(cols,index=df.index).replace([np.inf,-np.inf],np.nan)
    return idf.dropna(axis=1,how='all')


def shap_select_features(X_train, y_train, top_n, label=''):
    """SHAP pilote XGBoost — sélection par importance."""
    try:
        le  = LabelEncoder(); y_enc = le.fit_transform(y_train)
        X_c = (X_train.fillna(0).replace([np.inf,-np.inf],0)
               if hasattr(X_train,'fillna') else
               np.nan_to_num(X_train, nan=0., posinf=0., neginf=0.))
        pilot = XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.05,
                               subsample=0.8, eval_metric='mlogloss',
                               objective='multi:softprob',
                               random_state=SEED, n_jobs=-1)
        pilot.fit(X_c.values if hasattr(X_c,'values') else X_c, y_enc)
        expl  = shap.TreeExplainer(pilot)
        sv    = expl.shap_values((X_c.values if hasattr(X_c,'values') else X_c)[:500])
        if isinstance(sv,list):  arr = np.mean([np.abs(s) for s in sv],axis=0)
        elif np.array(sv).ndim==3: arr = np.abs(sv).mean(axis=2)
        else: arr = np.abs(sv)
        scores = pd.Series(arr.mean(axis=0),
                           index=X_c.columns if hasattr(X_c,'columns') else range(arr.shape[1]))
        top = scores.sort_values(ascending=False).head(top_n).index.tolist()
        if label: print(f"  [SHAP {label}] top-{top_n}/{len(scores)}")
        return top, scores
    except Exception as e:
        print(f"  [WARN SHAP] {e} — fallback MI")
        mi = mutual_info_classif(np.nan_to_num(X_train.values if hasattr(X_train,'values') else X_train,
                                                nan=0.,posinf=0.,neginf=0.), y_train, random_state=SEED)
        cols = X_train.columns if hasattr(X_train,'columns') else range(len(mi))
        scores = pd.Series(mi, index=cols)
        return scores.nlargest(top_n).index.tolist(), scores


def select_best_sampler(X_train, y_train):
    """Compare SMOTE/BorderlineSMOTE/SMOTETomek sur RandomForest pilote."""
    samplers = {
        'SMOTE':           SMOTE(random_state=SEED),
        'BorderlineSMOTE': BorderlineSMOTE(random_state=SEED, kind='borderline-1'),
        'SMOTETomek':      SMOTETomek(random_state=SEED),
    }
    pilot   = RandomForestClassifier(n_estimators=50, max_depth=4,
                                      random_state=SEED, n_jobs=-1)
    best_n, best_f1, best_s = 'SMOTE', -1, samplers['SMOTE']
    for name, samp in samplers.items():
        try:
            Xr,yr = samp.fit_resample(X_train, y_train)
            n = len(Xr); scores = []
            for fold in range(3):
                te = n*(fold+2)//4; ve = n*(fold+3)//4
                pilot.fit(Xr[:te], yr[:te])
                p = pilot.predict(Xr[te:ve])
                scores.append(f1_score(yr[te:ve],p,average='macro',zero_division=0))
            f = np.mean(scores)
            if f > best_f1: best_f1,best_n,best_s = f,name,samp
        except: pass
    print(f"  Sampler : {best_n} (F1={best_f1:.4f})")
    return best_n, best_s


## 7. Architectures DL : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT, Mamba

In [45]:
# ── LSTM ─────────────────────────────────────────────────────────────────────
class VIX_LSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2,
                 dropout=CONFIG['dropout'], n_classes=4):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(hidden_dim)
        self.fc   = nn.Sequential(nn.Linear(hidden_dim,64),nn.GELU(),nn.Dropout(dropout),
                                   nn.Linear(64,32),nn.GELU(),nn.Linear(32,n_classes))
    def forward(self,x):
        out,_ = self.lstm(x); return self.fc(self.norm(out[:,-1,:]))

# ── TCN ──────────────────────────────────────────────────────────────────────
class TCNBlock(nn.Module):
    def __init__(self,n_in,n_out,ks,dil,drop=0.2):
        super().__init__()
        pad = (ks-1)*dil
        self.conv = nn.utils.weight_norm(nn.Conv1d(n_in,n_out,ks,padding=pad,dilation=dil))
        self.drop = nn.Dropout(drop); self.act = nn.GELU()
        self.skip = nn.Conv1d(n_in,n_out,1) if n_in!=n_out else None
    def forward(self,x):
        out = self.act(self.drop(self.conv(x)[:,:,:-self.conv.padding[0]]
                                  if self.conv.padding[0]>0 else self.conv(x)))
        return self.act(out + (x if self.skip is None else self.skip(x)))

class VIX_TCN(nn.Module):
    def __init__(self,input_dim,channels=[32,64,128],ks=3,dropout=CONFIG['dropout'],n_classes=4):
        super().__init__()
        layers=[]; in_ch=input_dim
        for i,ch in enumerate(channels):
            layers.append(TCNBlock(in_ch,ch,ks,2**i,dropout)); in_ch=ch
        self.net=nn.Sequential(*layers)
        self.fc=nn.Sequential(nn.Linear(channels[-1],64),nn.GELU(),nn.Dropout(dropout),nn.Linear(64,n_classes))
    def forward(self,x): return self.fc(self.net(x.transpose(1,2)).mean(dim=2))

# ── Transformer ───────────────────────────────────────────────────────────────
class VIX_Transformer(nn.Module):
    def __init__(self,input_dim,d_model=128,nhead=4,num_layers=3,dropout=CONFIG['dropout'],n_classes=4):
        super().__init__()
        self.proj=nn.Linear(input_dim,d_model)
        self.pos_emb=nn.Embedding(CONFIG['lookback'],d_model)
        enc=nn.TransformerEncoderLayer(d_model,nhead,d_model*4,dropout,batch_first=True,norm_first=True)
        self.enc=nn.TransformerEncoder(enc,num_layers)
        self.fc=nn.Sequential(nn.Linear(d_model,64),nn.GELU(),nn.Dropout(dropout),nn.Linear(64,n_classes))
    def forward(self,x):
        B,T,_=x.shape; pos=torch.arange(T,device=x.device).unsqueeze(0).expand(B,-1)
        x=self.proj(x)+self.pos_emb(pos); return self.fc(self.enc(x)[:,-1,:])

# ── CNN-LSTM ──────────────────────────────────────────────────────────────────
class VIX_CNNLSTM(nn.Module):
    def __init__(self,input_dim,conv_f=64,lstm_h=128,ks=3,dropout=CONFIG['dropout'],n_classes=4):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv1d(input_dim,conv_f,ks,padding='same'),
                                  nn.GELU(),nn.BatchNorm1d(conv_f),nn.Dropout(dropout))
        self.lstm=nn.LSTM(conv_f,lstm_h,batch_first=True,num_layers=2,dropout=dropout)
        self.fc=nn.Sequential(nn.Linear(lstm_h,64),nn.GELU(),nn.Dropout(dropout),nn.Linear(64,n_classes))
    def forward(self,x): x=self.conv(x.transpose(1,2)).transpose(1,2); out,_=self.lstm(x); return self.fc(out[:,-1,:])

# ── N-BEATS ───────────────────────────────────────────────────────────────────
class NBeatsBlock(nn.Module):
    def __init__(self,input_size,theta_size,hidden=256):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(input_size,hidden),nn.ReLU(),nn.Linear(hidden,hidden),
                               nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,theta_size))
    def forward(self,x): return self.fc(x)

class VIX_NBeats(nn.Module):
    def __init__(self,input_dim,lookback,n_blocks=3,hidden=256,dropout=CONFIG['dropout'],n_classes=4):
        super().__init__()
        self.blocks=nn.ModuleList([NBeatsBlock(input_dim*lookback,hidden,hidden) for _ in range(n_blocks)])
        self.drop=nn.Dropout(dropout); self.fc=nn.Linear(hidden,n_classes)
    def forward(self,x):
        r=x.view(x.size(0),-1); out=None
        for b in self.blocks:
            t=self.drop(b(r)); out=t if out is None else out+t
        return self.fc(out)

# ── TFT ──────────────────────────────────────────────────────────────────────
class GLU(nn.Module):
    def __init__(self,d): super().__init__(); self.f=nn.Linear(d,d); self.g=nn.Linear(d,d)
    def forward(self,x): return self.f(x)*torch.sigmoid(self.g(x))

class VIX_TFT(nn.Module):
    def __init__(self,input_dim,d_model=128,nhead=4,lstm_layers=2,dropout=CONFIG['dropout'],n_classes=4):
        super().__init__()
        self.vsn=nn.Sequential(nn.Linear(input_dim,d_model),nn.GELU(),nn.Linear(d_model,input_dim),nn.Softmax(dim=-1))
        self.proj=nn.Linear(input_dim,d_model)
        self.lstm=nn.LSTM(d_model,d_model,num_layers=lstm_layers,batch_first=True,dropout=dropout)
        self.ln_lstm=nn.LayerNorm(d_model)
        enc=nn.TransformerEncoderLayer(d_model,nhead,d_model*2,dropout,batch_first=True,norm_first=True)
        self.attn=nn.TransformerEncoder(enc,num_layers=2)
        self.glu=GLU(d_model); self.ln=nn.LayerNorm(d_model)
        self.fc=nn.Sequential(nn.Linear(d_model,64),nn.GELU(),nn.Dropout(dropout),nn.Linear(64,n_classes))
    def forward(self,x):
        w=self.vsn(x.mean(dim=1,keepdim=True)); x=x*w; x=self.proj(x)
        lo,_=self.lstm(x); lo=self.ln_lstm(lo+x)
        ao=self.attn(lo); out=self.ln(self.glu(ao)+ao)
        return self.fc(out[:,-1,:])

# ── Mamba (SSM simplifié) ─────────────────────────────────────────────────────
class SSMLayer(nn.Module):
    def __init__(self,d,ks=21):
        super().__init__()
        self.kernel=nn.Parameter(torch.randn(d,1,ks)*0.01)
        self.norm=nn.LayerNorm(d); self.ig=nn.Linear(d,d); self.og=nn.Linear(d,d)
    def forward(self,x):
        B,L,D=x.shape; x_c=x.transpose(1,2); pad=self.kernel.shape[2]-1
        xp=F.pad(x_c,(pad,0)); k=torch.softmax(self.kernel,dim=-1)
        out=F.conv1d(xp,k,groups=D).transpose(1,2)
        out=out*torch.sigmoid(self.ig(x)); out=self.norm(out+x)
        return out*torch.sigmoid(self.og(out))

class VIX_Mamba(nn.Module):
    def __init__(self,input_dim,d_model=128,n_layers=4,dropout=CONFIG['dropout'],n_classes=4):
        super().__init__()
        self.proj=nn.Linear(input_dim,d_model)
        self.layers=nn.ModuleList([SSMLayer(d_model) for _ in range(n_layers)])
        self.drop=nn.Dropout(dropout); self.norm=nn.LayerNorm(d_model)
        self.fc=nn.Sequential(nn.Linear(d_model,64),nn.GELU(),nn.Dropout(dropout),nn.Linear(64,n_classes))
    def forward(self,x):
        x=self.proj(x)
        for l in self.layers: x=self.drop(l(x))
        return self.fc(self.norm(x[:,-1,:]))

VIX_MambaOfficial = VIX_Mamba  # fallback si mamba-ssm non disponible
print("7 architectures définies : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT, Mamba")


7 architectures définies : LSTM, TCN, Transformer, CNN-LSTM, N-BEATS, TFT, Mamba


## 8. Focal Loss, entraînement, évaluation

In [46]:
class FocalLoss(nn.Module):
    """Focal Loss multi-classes avec label smoothing. γ=2 réduit la contribution
    des exemples faciles pour focaliser sur les classes difficiles (UP_FORT, DOWN_FORT)."""
    def __init__(self,gamma=2.,alpha=None,label_smoothing=0.1):
        super().__init__(); self.gamma=gamma; self.alpha=alpha; self.ls=label_smoothing
    def forward(self,logits,targets):
        n_cls=logits.size(1)
        oh=torch.zeros_like(logits).scatter_(1,targets.unsqueeze(1),1)
        smooth=oh*(1-self.ls)+self.ls/n_cls
        lp=torch.log_softmax(logits,dim=1); p=lp.exp()
        at=self.alpha.to(logits.device)[targets].unsqueeze(1) if self.alpha is not None else 1.
        return (-(at*(1-p)**self.gamma*smooth*lp).sum(dim=1)).mean()


def compute_class_weights(y,n=4):
    c=np.bincount(y,minlength=n); w=1./(c+1e-6); return torch.tensor(w/w.sum()*n,dtype=torch.float32)


def train_model(model,dl_tr,dl_val,class_weights=None,epochs=CONFIG['epochs'],
                lr=CONFIG['lr'],wd=CONFIG['weight_decay'],label='',patience=10):
    crit  = FocalLoss(gamma=2.,alpha=class_weights,label_smoothing=0.1)
    opt   = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt,T_0=10,T_mult=2)
    best_loss,best_state,wait = float('inf'),None,0
    t0 = time.time()
    for ep in range(epochs):
        model.train()
        for bx,by in dl_tr:
            bx,by=bx.to(device),by.to(device); opt.zero_grad()
            loss=crit(model(bx),by); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step()
        sched.step()
        model.eval(); vl=0.
        with torch.no_grad():
            for bx,by in dl_val: bx,by=bx.to(device),by.to(device); vl+=crit(model(bx),by).item()
        if vl<best_loss: best_loss=vl; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else:
            wait+=1
            if wait>=patience: print(f"  [{label}] Early stop ep{ep+1} ({time.time()-t0:.1f}s)"); break
        if (ep+1)%10==0: print(f"  [{label}] ep{ep+1} val_loss={vl:.4f} ({time.time()-t0:.1f}s)")
    if best_state: model.load_state_dict(best_state)


def evaluate_model(model,loader,label=''):
    model.eval(); preds,targets=[],[]
    with torch.no_grad():
        for bx,by in loader:
            preds.extend(model(bx.to(device)).argmax(1).cpu().numpy()); targets.extend(by.numpy())
    y,p=np.array(targets),np.array(preds)
    dm={0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    yd=[dm[i] for i in y]; pd_=[dm[i] for i in p]
    m={'F1_4cls':f1_score(y,p,average='macro',labels=[0,1,2,3],zero_division=0),
       'Acc_4cls':accuracy_score(y,p),
       'Acc_dir':accuracy_score(yd,pd_),
       'F1_dir':f1_score(yd,pd_,average='macro',zero_division=0),
       'F1_UP':f1_score(yd,pd_,pos_label='UP',average='binary',zero_division=0),
       'F1_DOWN':f1_score(yd,pd_,pos_label='DOWN',average='binary',zero_division=0)}
    ui=[i for i,v in enumerate(y) if dm[v]=='UP']
    di=[i for i,v in enumerate(y) if dm[v]=='DOWN']
    if ui:
        yt=['FORT' if y[i]==3 else 'FAIBLE' for i in ui]; yp=['FORT' if p[i]==3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT']=f1_score(yt,yp,pos_label='FORT',average='binary',zero_division=0)
    if di:
        yt=['FORT' if y[i]==0 else 'FAIBLE' for i in di]; yp=['FORT' if p[i]==0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT']=f1_score(yt,yp,pos_label='FORT',average='binary',zero_division=0)
    if label: print(f"  [{label}] F1_dir={m['F1_dir']:.4f} F1_UP_FORT={m.get('F1_UP_FORT',0):.4f} F1_DOWN_FORT={m.get('F1_DOWN_FORT',0):.4f}")
    return m


## 9. Temperature Scaling — Calibration des probabilités

In [47]:
class TemperatureScaler(nn.Module):
    """
    Module de calibration par Temperature Scaling.
    Enveloppe un modèle PyTorch existant et apprend le paramètre T.

    Usage :
        scaler = TemperatureScaler(trained_model)
        scaler.calibrate(val_loader, device)
        probs = scaler.predict_proba(X_tensor)
    """
    def __init__(self, model):
        super().__init__()
        self.model = model
        # T initialisé à 1.0 (pas de calibration) — appris sur le val set
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, x):
        logits = self.model(x)
        return logits / self.temperature

    def calibrate(self, val_loader, device, lr=CONFIG['temperature_lr'],
                  epochs=CONFIG['temperature_epochs']):
        """Apprend T en minimisant la NLL sur le val set."""
        self.to(device)
        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=epochs)
        nll_criterion = nn.CrossEntropyLoss()

        # Collecter tous les logits et targets du val set (une seule fois)
        all_logits, all_targets = [], []
        self.model.eval()
        with torch.no_grad():
            for bx, by in val_loader:
                all_logits.append(self.model(bx.to(device)))
                all_targets.append(by.to(device))
        all_logits  = torch.cat(all_logits)
        all_targets = torch.cat(all_targets)

        def eval_fn():
            optimizer.zero_grad()
            loss = nll_criterion(all_logits / self.temperature, all_targets)
            loss.backward()
            return loss

        optimizer.step(eval_fn)
        print(f"  [TemperatureScaling] T optimal = {self.temperature.item():.4f}")

    @torch.no_grad()
    def predict_proba(self, x_tensor):
        self.eval()
        logits = self.forward(x_tensor.to(next(self.parameters()).device))
        return torch.softmax(logits, dim=1).cpu().numpy()


def plot_reliability_diagram(y_true, y_prob_uncal, y_prob_cal, n_bins=10, label=''):
    """
    Reliability diagram avant/après calibration.
    Chaque bin = observations dont la confiance du modèle est dans [b, b+1/n_bins).
    La courbe idéale est la diagonale (confiance = précision réelle).
    L'ECE (Expected Calibration Error) mesure l'aire entre la courbe et la diagonale.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, probs, title in zip(axes,
                                  [y_prob_uncal, y_prob_cal],
                                  ['Avant calibration', 'Après Temperature Scaling']):
        # Classe la plus probable
        conf = probs.max(axis=1)
        correct = (probs.argmax(axis=1) == y_true).astype(float)

        bins = np.linspace(0, 1, n_bins + 1)
        bin_confs, bin_accs, bin_sizes = [], [], []
        for i in range(n_bins):
            mask = (conf >= bins[i]) & (conf < bins[i+1])
            if mask.sum() > 0:
                bin_confs.append(conf[mask].mean())
                bin_accs.append(correct[mask].mean())
                bin_sizes.append(mask.sum())

        ece = sum(s * abs(c - a) for s,c,a in zip(bin_sizes,bin_confs,bin_accs)) / len(y_true)

        ax.bar(bin_confs, bin_accs, width=0.08, alpha=0.7, color='steelblue', label='Modèle')
        ax.plot([0,1],[0,1],'r--', label='Calibration parfaite')
        ax.set_xlabel('Confiance'); ax.set_ylabel('Précision réelle')
        ax.set_title(f'{title}\nECE = {ece:.4f} {label}')
        ax.legend()
    plt.tight_layout(); plt.show()
    return ece


## 10. Stacking DL+ML + Threshold Optimization

In [48]:
class MetaStackingClassifier:
    """
    Méta-classifieur par stacking.

    Niveau 1 : dict de modèles DL PyTorch (déjà entraînés)
    Niveau 2 : XGBoost entraîné sur les probabilités OOF du niveau 1

    La stratégie OOF (Out-of-Fold) garantit l'absence de leakage :
    les features du méta-modèle sont générées par les modèles niveau 1
    sur des données qu'ils n'ont pas vues pendant leur entraînement.
    """
    def __init__(self, base_models: dict, meta_model=None, n_folds=5):
        self.base_models = base_models  # {nom: modèle PyTorch}
        self.meta_model  = meta_model or XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric='mlogloss', random_state=SEED, n_jobs=-1
        )
        self.n_folds = n_folds
        self.fitted  = False
        self.thresholds_by_regime = {}

    def _get_proba(self, model, loader, device):
        """Extrait les probabilités softmax d'un modèle PyTorch sur un DataLoader."""
        model.eval()
        probs, targets = [], []
        with torch.no_grad():
            for bx, by in loader:
                logits = model(bx.to(device))
                probs.append(torch.softmax(logits, dim=1).cpu().numpy())
                targets.append(by.numpy())
        return np.vstack(probs), np.concatenate(targets)

    def generate_oof_features(self, X_train_seq, y_train, device):
        """
        Génère les features OOF pour le méta-modèle.
        Pour chaque fold temporel, les modèles sont évalués sur le fold
        qu'ils n'ont pas vu — garantit l'absence de leakage.
        """
        N = len(X_train_seq)
        n_models = len(self.base_models)
        oof_probs = np.zeros((N, n_models * 4))  # 4 classes par modèle

        tscv = TimeSeriesSplit(n_splits=self.n_folds)
        for fold_idx, (tr_idx, val_idx) in enumerate(tscv.split(X_train_seq)):
            if len(val_idx) < 10: continue
            x_val = torch.tensor(X_train_seq[val_idx], dtype=torch.float32)
            if x_val.dim() == 2:
                x_val = x_val.unsqueeze(1).repeat(1, CONFIG['lookback'], 1)
            y_val = torch.tensor(y_train[val_idx], dtype=torch.long)
            ds_val = TensorDataset(x_val, y_val)
            dl_val = DataLoader(ds_val, batch_size=256)

            for m_idx, (name, model) in enumerate(self.base_models.items()):
                probs, _ = self._get_proba(model, dl_val, device)
                oof_probs[val_idx, m_idx*4:(m_idx+1)*4] = probs

            if (fold_idx + 1) % 2 == 0:
                print(f"  [Stacking OOF] Fold {fold_idx+1}/{self.n_folds} ✓")

        return oof_probs

    def fit(self, X_train_seq, y_train, device):
        print("  [Stacking] Génération features OOF...")
        oof_features = self.generate_oof_features(X_train_seq, y_train, device)
        print(f"  [Stacking] Entraînement méta-modèle (XGBoost) sur {oof_features.shape}...")
        self.meta_model.fit(oof_features, y_train)
        self.fitted = True
        print("  [Stacking] Fit terminé.")

    def predict_proba(self, X_test_seq, device):
        """
        Test : chaque modèle de niveau 1 prédit sur tout le test,
        les probabilités sont concatenées et passées au méta-modèle.
        """
        assert self.fitted
        test_probs_list = []
        for name, model in self.base_models.items():
            x_t = torch.tensor(X_test_seq, dtype=torch.float32)
            if x_t.dim() == 2:
                x_t = x_t.unsqueeze(1).repeat(1, CONFIG['lookback'], 1)
            ds  = TensorDataset(x_t, torch.zeros(len(x_t), dtype=torch.long))
            dl  = DataLoader(ds, batch_size=256)
            probs, _ = self._get_proba(model, dl, device)
            test_probs_list.append(probs)
        meta_features = np.hstack(test_probs_list)
        return self.meta_model.predict_proba(meta_features)

    def predict(self, X_test_seq, device):
        return self.predict_proba(X_test_seq, device).argmax(axis=1)


def optimize_direction_threshold(probs, y_true, regime_mask=None, n_steps=50):
    """
    Optimise le seuil de décision UP/DOWN sur le val set pour maximiser F1_dir.

    probs      : (N, 4) probabilités softmax
    y_true     : (N,) labels 0-3
    regime_mask: masque optionnel pour optimiser par régime

    Retourne le seuil optimal τ* ∈ [0.3, 0.7]
    """
    dir_map   = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    # P(UP) = sum des probabilités des classes UP
    p_up      = probs[:, 2] + probs[:, 3]
    yd_true   = [dir_map[y] for y in y_true]

    thresholds = np.linspace(0.3, 0.7, n_steps)
    best_thr, best_f1 = 0.5, -1

    for thr in thresholds:
        yd_pred = ['UP' if p > thr else 'DOWN' for p in p_up]
        if regime_mask is not None:
            yd_true_r = [yd_true[i] for i in range(len(yd_true)) if regime_mask[i]]
            yd_pred_r = [yd_pred[i] for i in range(len(yd_pred)) if regime_mask[i]]
        else:
            yd_true_r, yd_pred_r = yd_true, yd_pred
        f1 = f1_score(yd_true_r, yd_pred_r, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr

    return best_thr, best_f1


## 11. Options Flow et Corrélation Implicite

In [49]:
def build_options_flow_features(df_raw, train_end_idx):
    """
    Construit des proxies du flux d'options depuis les données disponibles.

    PCR direct : téléchargé depuis CBOE via FRED (PUTCALLRATIO)
    ou estimé depuis les volumes VIX/VVIX si non disponible.
    """
    feats = pd.DataFrame(index=df_raw.index)
    t0 = time.time()

    # ── Tentative de téléchargement du PCR CBOE via FRED ─────────────────────
    # PUTCALLRATIO : ratio Put/Call total, disponible sur FRED
    try:
        pcr = web.DataReader('PUTCALLRATIO', 'fred',
                              start=CONFIG['start_date']).squeeze()
        pcr = pcr.reindex(df_raw.index, method='ffill')
        feats['pcr_total']    = pcr
        feats['pcr_zscore']   = (pcr - pcr.iloc[:train_end_idx].mean()) / pcr.iloc[:train_end_idx].std()
        feats['pcr_ma5']      = pcr.rolling(5, min_periods=3).mean()
        feats['pcr_spike']    = (pcr - feats['pcr_ma5']) / feats['pcr_ma5'].replace(0, np.nan)
        print(f"  [PCR] CBOE Put/Call Ratio chargé depuis FRED ({time.time()-t0:.1f}s)")
    except Exception as e:
        print(f"  [PCR] FRED non disponible ({e}) — proxy via VIX/VVIX")
        # Proxy : ratio VVIX/VIX comme indicateur de demande d'options
        vix_cols  = [c for c in df_raw.columns if 'VIX' in c and 'VVIX' not in c]
        vvix_cols = [c for c in df_raw.columns if 'VVIX' in c]
        if vix_cols and vvix_cols:
            vix  = df_raw[vix_cols[0]].ffill()
            vvix = df_raw[vvix_cols[0]].ffill()
            pcr_proxy = vvix / vix.replace(0, np.nan)
            feats['pcr_proxy']       = pcr_proxy
            feats['pcr_proxy_zscore']= (pcr_proxy - pcr_proxy.iloc[:train_end_idx].mean()) /                                         pcr_proxy.iloc[:train_end_idx].std()

    # ── Corrélation implicite (proxy via secteurs ETF) ────────────────────────
    # σ_indice = VIX/100 (volatilité implicite SPX annualisée)
    # σ_secteur = vol réalisée rolling 21j des ETFs sectoriels
    # Poids égaux (simplification — un vrai calcul utiliserait les poids de l'indice)
    sector_etfs = [c for c in df_raw.columns
                   if any(s in c for s in ['XLK','XLF','XLE','XLV','XLU','XLB','XLI','XLY'])]
    vix_col = [c for c in df_raw.columns if 'IDX_VIX' in c or (c.endswith('VIX') and 'VXN' not in c and 'VVIX' not in c)]

    if sector_etfs and vix_col:
        vix  = df_raw[vix_col[0]].ffill() / 100  # en décimal
        w    = 1.0 / len(sector_etfs)

        # Volatilités réalisées des secteurs (rolling 21j)
        sector_vols = {}
        for col in sector_etfs:
            ret = np.log(df_raw[col].ffill() / df_raw[col].ffill().shift(1))
            sector_vols[col] = ret.rolling(21, min_periods=10).std() * np.sqrt(252)

        # Numérateur : σ_indice² - Σ wᵢ²σᵢ²
        sum_wi2_sigma2 = sum(w**2 * sv**2 for sv in sector_vols.values())
        # Dénominateur : Σᵢ≠ⱼ wᵢwⱼσᵢσⱼ (approximé par σ_indice² - sum_wi2_sigma2)
        numerator = vix**2 - sum_wi2_sigma2
        # Dénominateur : somme des covariances croisées attendues
        n = len(sector_etfs)
        avg_sigma  = pd.concat(sector_vols.values(), axis=1).mean(axis=1)
        denominator = (n**2 - n) * w**2 * avg_sigma**2

        impl_corr = (numerator / denominator.replace(0, np.nan)).clip(-1, 1)
        impl_corr_smooth = impl_corr.rolling(5, min_periods=3).mean()

        feats['impl_corr']        = impl_corr_smooth
        feats['impl_corr_zscore'] = (impl_corr_smooth - impl_corr_smooth.iloc[:train_end_idx].mean()) /                                      impl_corr_smooth.iloc[:train_end_idx].std()
        feats['impl_corr_delta']  = impl_corr_smooth.diff()
        print(f"  [Impl Corr] Corrélation implicite calculée sur {len(sector_etfs)} secteurs ({time.time()-t0:.1f}s)")

    return feats.replace([np.inf, -np.inf], np.nan)


## 12. Conformal Prediction

In [50]:
class TemporalConformalClassifier:
    """
    Conformal Prediction adapté aux séries temporelles.

    Différence vs conformal classique : en finance, les observations ne sont pas
    échangeables (il y a une dépendance temporelle). On utilise une variante
    où le cal set est **chronologiquement postérieur** au train, pour respecter
    la causalité temporelle.

    Référence :
    - Vovk et al. (1999) — Conformal Prediction
    - Barber et al. (2023) — Conformal Prediction for Time Series
    """
    def __init__(self, alpha=CONFIG['conformal_alpha']):
        self.alpha = alpha
        self.q_hat = None  # quantile de calibration

    def calibrate(self, model, cal_loader, device):
        """
        Calcule les non-conformity scores sur l'ensemble de calibration.

        score_i = 1 - P(y_i | x_i)  (plus le score est élevé, plus la prédiction est 'surprenante')
        q_hat = quantile (1-alpha) des scores → seuil de l'ensemble de confiance
        """
        model.eval()
        scores = []
        with torch.no_grad():
            for bx, by in cal_loader:
                logits = model(bx.to(device))
                probs  = torch.softmax(logits, dim=1).cpu().numpy()
                for i, label in enumerate(by.numpy()):
                    # Non-conformity score : 1 - probabilité de la vraie classe
                    scores.append(1.0 - probs[i, label])

        scores   = np.array(scores)
        n        = len(scores)
        # Quantile ajusté pour la garantie de couverture finie
        q_level  = np.ceil((n + 1) * (1 - self.alpha)) / n
        self.q_hat = float(np.quantile(scores, min(q_level, 1.0)))
        print(f"  [Conformal] q_hat = {self.q_hat:.4f} (alpha={self.alpha}, n_cal={n})")

    def predict_set(self, model, x_tensor, device):
        """
        Prédit l'ensemble de confiance C(x) pour chaque observation.
        C(x) = {k : 1 - P(k|x) <= q_hat}

        Retourne une liste de listes : chaque sous-liste contient les classes
        plausibles pour l'observation correspondante.
        """
        assert self.q_hat is not None, "Calibrer d'abord avec .calibrate()"
        model.eval()
        with torch.no_grad():
            logits = model(x_tensor.to(device))
            probs  = torch.softmax(logits, dim=1).cpu().numpy()

        prediction_sets = []
        for i in range(len(probs)):
            # Inclure toutes les classes dont le non-conformity score <= q_hat
            conf_set = [k for k in range(4) if (1 - probs[i, k]) <= self.q_hat]
            prediction_sets.append(conf_set)

        return prediction_sets

    def evaluate_coverage(self, model, test_loader, device):
        """
        Vérifie empiriquement que la couverture est bien >= 1-alpha.
        Corollaire : si coverage < 1-alpha, il y a un problème (données non-échangeables
        ou calibration insuffisante).
        """
        covered = 0
        total   = 0
        set_sizes = []

        model.eval()
        with torch.no_grad():
            for bx, by in test_loader:
                x_t   = bx.to(device)
                logits = model(x_t)
                probs  = torch.softmax(logits, dim=1).cpu().numpy()
                labels = by.numpy()

                for i, label in enumerate(labels):
                    conf_set = [k for k in range(4) if (1 - probs[i, k]) <= self.q_hat]
                    if label in conf_set:
                        covered += 1
                    set_sizes.append(len(conf_set))
                    total += 1

        coverage   = covered / total
        avg_set_sz = np.mean(set_sizes)
        print(f"  [Conformal] Couverture empirique : {coverage:.4f} (cible : {1-self.alpha:.2f})")
        print(f"  [Conformal] Taille moyenne des ensembles : {avg_set_sz:.2f} / 4 classes")
        return coverage, avg_set_sz

    def trading_signal(self, prediction_set):
        """
        Convertit l'ensemble de confiance en signal de trading.
        Plus l'ensemble est petit et concentré sur des classes extrêmes,
        plus le signal est fort.
        """
        CLASS_LABELS = ['DOWN_FORT','DOWN_FAIBLE','UP_FAIBLE','UP_FORT']
        set_labels = [CLASS_LABELS[k] for k in prediction_set]
        n = len(prediction_set)

        if n == 1:
            return {'signal': set_labels[0], 'confidence': 'FORTE', 'sizing': 1.0}
        elif n == 2:
            if all(k >= 2 for k in prediction_set):
                return {'signal': 'UP',   'confidence': 'MODÉRÉE', 'sizing': 0.5}
            elif all(k < 2  for k in prediction_set):
                return {'signal': 'DOWN', 'confidence': 'MODÉRÉE', 'sizing': 0.5}
        elif n >= 3:
            return {'signal': 'INCERTAIN', 'confidence': 'FAIBLE', 'sizing': 0.0}
        return {'signal': 'NEUTRE', 'confidence': 'NULLE', 'sizing': 0.0}


## 13. Adversarial Validation

In [51]:
def adversarial_validation(X_train, X_test, feature_names=None,
                            threshold=CONFIG['adv_val_threshold'],
                            n_estimators=100):
    """
    Validation adversariale : détecte le drift de distribution entre train et test.

    Algorithme :
    1. Créer dataset binaire {train:0, test:1}
    2. Entraîner RandomForest pour distinguer les deux
    3. AUC proche de 0.5 = pas de drift / proche de 1.0 = drift sévère

    Paramètres
    ----------
    X_train : array (N_train, F) — features du train
    X_test  : array (N_test, F)  — features du test
    threshold: AUC au-dessus duquel on signale un drift

    Retourne
    --------
    auc         : float — AUC du classifieur adversarial
    drift_feats : list — features les plus responsables du drift (SHAP)
    """
    t0 = time.time()

    # Sous-échantillonner pour équilibrer les classes
    n_min = min(len(X_train), len(X_test))
    idx_tr = np.random.choice(len(X_train), n_min, replace=False)
    idx_te = np.random.choice(len(X_test),  n_min, replace=False)

    X_adv = np.vstack([X_train[idx_tr], X_test[idx_te]])
    y_adv = np.array([0]*n_min + [1]*n_min)

    # Validation croisée temporelle (pas de shuffle — respecter la causalité)
    tscv  = TimeSeriesSplit(n_splits=5)
    aucs  = []
    clf   = RandomForestClassifier(n_estimators=n_estimators, max_depth=5,
                                    random_state=SEED, n_jobs=-1)

    for tr_idx, val_idx in tscv.split(X_adv):
        clf.fit(X_adv[tr_idx], y_adv[tr_idx])
        probs = clf.predict_proba(X_adv[val_idx])
        if probs.shape[1] < 2:
            aucs.append(0.5)
            continue
        probs = probs[:, 1]
        auc   = roc_auc_score(y_adv[val_idx], probs)
        aucs.append(auc)

    mean_auc = np.mean(aucs)

    print(f"\n  [Adversarial Validation] AUC = {mean_auc:.4f} ({time.time()-t0:.1f}s)")
    if mean_auc > threshold:
        print(f"  [ALERTE] Drift significatif détecté (AUC > {threshold})")
        print(f"  Action suggérée : restreindre la fenêtre de train aux données récentes")
    else:
        print(f"  [OK] Pas de drift significatif (AUC <= {threshold})")

    # Identifier les features responsables du drift
    drift_feats = []
    if feature_names is not None:
        # Entraîner sur tout le dataset pour l'analyse SHAP
        clf.fit(X_adv, y_adv)
        importances = pd.Series(clf.feature_importances_, index=feature_names)
        top_drift = importances.nlargest(10)
        drift_feats = top_drift.index.tolist()

        print(f"  Top features responsables du drift :")
        for feat, imp in top_drift.items():
            print(f"    {feat:<45} importance = {imp:.4f}")

    return mean_auc, drift_feats


def plot_distribution_shift(X_train, X_test, feature_names, n_features=6):
    """
    Visualise le shift de distribution pour les N features les plus importantes.
    Utile pour comprendre concrètement comment la distribution a changé.
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, feat in enumerate(feature_names[:n_features]):
        if feat not in [feature_names[j] for j in range(len(feature_names))]:
            continue
        feat_idx = list(feature_names).index(feat)
        ax = axes[i]
        ax.hist(X_train[:, feat_idx], bins=30, alpha=0.6, label='Train', color='blue', density=True)
        ax.hist(X_test[:,  feat_idx], bins=30, alpha=0.6, label='Test',  color='red',  density=True)
        ax.set_title(f'{feat[:30]}')
        ax.legend()

    plt.suptitle('Distribution Shift : Train vs Test', fontsize=14)
    plt.tight_layout()
    plt.show()


## 14. Stress Testing sectoriel

In [52]:
def stress_test_models(models_dict, df_full, feature_cols, target_col,
                       scaler, train_end_date, lookback=CONFIG['lookback']):
    """
    Évalue chaque modèle sur les grandes périodes de crise historiques.

    Paramètres
    ----------
    models_dict  : dict {nom: modèle PyTorch calibré}
    df_full      : DataFrame complet (features + target)
    feature_cols : colonnes de features
    train_end_date : date de fin du train (pour s'assurer qu'on ne teste que sur le test)
    """
    results = {}

    print("\n" + "="*60)
    print("STRESS TESTING — Performance sur les grandes crises")
    print("="*60)

    for crisis_name, (start, end) in CRISIS_PERIODS.items():
        # Vérifier que la crise est dans le test set
        crisis_start = pd.Timestamp(start)
        crisis_end   = pd.Timestamp(end)
        train_end    = pd.Timestamp(train_end_date)

        if crisis_end <= train_end:
            print(f"  [SKIP] {crisis_name} : antérieure à la fin du train")
            continue

        # Données de crise (subset du test)
        crisis_mask = ((df_full.index >= crisis_start) &
                        (df_full.index <= crisis_end) &
                        (df_full.index > train_end))
        df_crisis = df_full.loc[crisis_mask].dropna(subset=[target_col])

        if len(df_crisis) < 10:
            print(f"  [SKIP] {crisis_name} : moins de 10 observations")
            continue

        y_crisis = df_crisis[target_col].values.astype(int)
        X_crisis = scaler.transform(df_crisis[feature_cols].fillna(0))

        # Créer les séquences lookback
        if len(X_crisis) <= lookback:
            print(f"  [SKIP] {crisis_name} : trop peu d'observations pour le lookback")
            continue

        # Construire le tenseur de séquences
        X_seq = np.array([X_crisis[i:i+lookback] for i in range(len(X_crisis)-lookback)])
        y_seq = y_crisis[lookback:]
        x_t   = torch.tensor(X_seq, dtype=torch.float32)

        results[crisis_name] = {'n_obs': len(y_seq)}
        print(f"\n  📉 {crisis_name} ({start} → {end}) — {len(y_seq)} obs")

        for model_name, model in models_dict.items():
            model.eval()
            with torch.no_grad():
                logits = model(x_t.to(device))
                preds  = logits.argmax(1).cpu().numpy()

            # Métriques hiérarchiques
            dir_map  = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
            yd_true  = [dir_map[y] for y in y_seq]
            yd_pred  = [dir_map[p] for p in preds]
            f1_dir   = f1_score(yd_true, yd_pred, average='macro', zero_division=0)
            acc_dir  = accuracy_score(yd_true, yd_pred)

            up_idx = [i for i,y in enumerate(y_seq) if dir_map[y]=='UP']
            f1_uf  = 0.0
            if up_idx:
                yt_up = ['FORT' if y_seq[i]==3 else 'FAIBLE' for i in up_idx]
                yp_up = ['FORT' if preds[i]==3 else 'FAIBLE' for i in up_idx]
                f1_uf = f1_score(yt_up,yp_up,pos_label='FORT',average='binary',zero_division=0)

            results[crisis_name][model_name] = {
                'f1_dir': f1_dir, 'acc_dir': acc_dir, 'f1_up_fort': f1_uf
            }
            print(f"    {model_name:<20} F1_dir={f1_dir:.3f}  Acc={acc_dir:.3f}  F1_UP_FORT={f1_uf:.3f}")

    # Résumé comparatif
    print("\n" + "="*60)
    print("RÉSUMÉ STRESS TEST — F1_dir par crise et modèle")
    print("="*60)
    crisis_df_rows = []
    for crisis, data in results.items():
        for model_name in models_dict:
            if model_name in data:
                crisis_df_rows.append({
                    'Crisis': crisis, 'Model': model_name,
                    'F1_dir': data[model_name]['f1_dir'],
                    'F1_UP_FORT': data[model_name]['f1_up_fort'],
                    'N_obs': data['n_obs']
                })
    if crisis_df_rows:
        df_stress = pd.DataFrame(crisis_df_rows)
        pivot = df_stress.pivot(index='Crisis', columns='Model', values='F1_dir')
        print(pivot.round(3).to_string())

    return results


## 15. Ensemble Asymétrique

In [53]:
class AsymmetricEnsemble:
    """
    Ensemble dont les poids varient selon le régime de marché courant.

    Pour chaque régime (CALM/NORMAL/STRESS), apprend un vecteur de poids
    qui maximise le F1_dir sur le val set filtré par ce régime.

    La sélection du régime est basée sur le niveau de VIX :
    - VIX < REGIME_THRESHOLDS['calm']   → CALM
    - VIX >= REGIME_THRESHOLDS['stress'] → STRESS
    - Sinon                               → NORMAL
    """
    def __init__(self, models_dict: dict):
        self.models = models_dict
        self.weights_by_regime = {
            'CALM':   np.ones(len(models_dict)) / len(models_dict),  # uniform init
            'NORMAL': np.ones(len(models_dict)) / len(models_dict),
            'STRESS': np.ones(len(models_dict)) / len(models_dict),
        }
        self.fitted = False

    def _get_vix_regime(self, vix_value):
        if vix_value < REGIME_THRESHOLDS['calm']:
            return 'CALM'
        elif vix_value >= REGIME_THRESHOLDS['stress']:
            return 'STRESS'
        return 'NORMAL'

    def _get_all_proba(self, loader, device):
        """Collecte les probabilités de tous les modèles sur un DataLoader."""
        all_probs = {name: [] for name in self.models}
        all_targets = []

        for bx, by in loader:
            bx = bx.to(device)
            for name, model in self.models.items():
                model.eval()
                with torch.no_grad():
                    probs = torch.softmax(model(bx), dim=1).cpu().numpy()
                all_probs[name].append(probs)
            all_targets.extend(by.numpy())

        return {k: np.vstack(v) for k, v in all_probs.items()}, np.array(all_targets)

    def fit_regime_weights(self, val_loader, vix_val_series, device):
        """
        Apprend les poids optimaux par régime sur le val set.
        Minimise la NLL (ou maximise F1_dir) par régime.
        """
        print("  [Ensemble Asymétrique] Optimisation des poids par régime...")
        all_probs_dict, y_val = self._get_all_proba(val_loader, device)
        dir_map = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}

        for regime in ['CALM', 'NORMAL', 'STRESS']:
            # Identifier les observations de ce régime
            if vix_val_series is not None and len(vix_val_series) == len(y_val):
                regime_mask = np.array([
                    self._get_vix_regime(v) == regime
                    for v in vix_val_series.values
                ])
            else:
                # Fallback : régime par tertile de l'index
                regime_mask = np.ones(len(y_val), dtype=bool)

            if regime_mask.sum() < 10:
                print(f"    {regime}: trop peu d'obs ({regime_mask.sum()}), poids uniformes")
                continue

            y_reg = y_val[regime_mask]
            yd_true_reg = [dir_map[y] for y in y_reg]

            # Optimisation par recherche sur grille simple (Dirichlet sampling)
            best_weights = np.ones(len(self.models)) / len(self.models)
            best_f1 = -1

            # 200 combinaisons aléatoires de poids
            np.random.seed(SEED)
            for _ in range(200):
                # Tirer des poids via Dirichlet (distribués sur le simplex)
                w = np.random.dirichlet(np.ones(len(self.models)))
                # Ensemble pondéré
                ensemble_probs = sum(
                    w[i] * all_probs_dict[name][regime_mask]
                    for i, name in enumerate(self.models)
                )
                yd_pred = [dir_map[p] for p in ensemble_probs.argmax(axis=1)]
                f1 = f1_score(yd_true_reg, yd_pred, average='macro', zero_division=0)
                if f1 > best_f1:
                    best_f1, best_weights = f1, w

            self.weights_by_regime[regime] = best_weights
            print(f"    {regime}: F1_dir = {best_f1:.4f} | poids = {dict(zip(self.models.keys(), best_weights.round(3)))}")

        self.fitted = True

    def predict_proba(self, x_tensor, vix_value, device):
        """
        Prédit les probabilités avec les poids du régime courant.

        vix_value : niveau de VIX du jour de la prédiction
        """
        regime  = self._get_vix_regime(vix_value)
        weights = self.weights_by_regime[regime]

        ensemble_probs = np.zeros((len(x_tensor), 4))
        for i, (name, model) in enumerate(self.models.items()):
            model.eval()
            with torch.no_grad():
                probs = torch.softmax(model(x_tensor.to(device)), dim=1).cpu().numpy()
            ensemble_probs += weights[i] * probs

        return ensemble_probs, regime

    def evaluate(self, test_loader, vix_test_series, device):
        """Évalue l'ensemble asymétrique en utilisant le régime de chaque jour."""
        all_preds, all_targets = [], []
        dir_map = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}

        for batch_idx, (bx, by) in enumerate(test_loader):
            # VIX moyen du batch pour déterminer le régime
            batch_size = len(bx)
            start_idx  = batch_idx * test_loader.batch_size
            end_idx    = min(start_idx + batch_size, len(vix_test_series))
            if end_idx > start_idx and vix_test_series is not None:
                vix_val = vix_test_series.iloc[start_idx:end_idx].mean()
            else:
                vix_val = 20.0  # NORMAL par défaut

            probs, regime = self.predict_proba(bx, vix_val, device)
            all_preds.extend(probs.argmax(axis=1))
            all_targets.extend(by.numpy())

        yd_t = [dir_map[y] for y in all_targets]
        yd_p = [dir_map[p] for p in all_preds]
        f1   = f1_score(yd_t, yd_p, average='macro', zero_division=0)
        print(f"  [Ensemble Asymétrique] F1_dir = {f1:.4f}")
        return f1


## 16. Pipeline V2 — Intégration complète

In [56]:
def run_pipeline_v2(df_raw, horizon=5):
    """Pipeline V2 complet. Exécuter après avoir défini df_raw."""
    t_total = time.time()
    print(f"\n{'='*60}\nPIPELINE V2 | h={horizon}j\n{'='*60}")

    # ── Identifier colonnes VIX et SPX ───────────────────────────────────────
    vix_col  = [c for c in df_raw.columns if ('IDX_VIX' in c or c.endswith('_VIX'))
                 and 'VXN' not in c and 'VVIX' not in c][0]
    spx_cols = [c for c in df_raw.columns if 'GSPC' in c or c=='SPY']
    spx_col  = spx_cols[0] if spx_cols else None
    vix_series = df_raw[vix_col].ffill()

    # ── Split 70/10/20 (train/val/test) ──────────────────────────────────────
    all_dates   = df_raw.dropna(how='all').index.sort_values()
    n           = len(all_dates)
    train_end   = all_dates[int(n*0.70)]
    val_end     = all_dates[int(n*0.80)]
    train_end_idx = int(n*0.70)
    print(f"  Train  → {train_end.date()} | Val → {val_end.date()} | Test → {all_dates[-1].date()}")

    # ── Features ─────────────────────────────────────────────────────────────
    print(f"  [Features TS] ({time.time()-t_total:.1f}s)")
    ts_feats  = build_ts_features(df_raw, train_end_idx)
    spx_s     = df_raw[spx_col].ffill() if spx_col else pd.Series(1., index=df_raw.index)
    adv_feats = build_advanced_features(df_raw, vix_series, spx_s, train_end_idx)
    print(f"  [Options Flow] ({time.time()-t_total:.1f}s)")
    opt_feats = build_options_flow_features(df_raw, train_end_idx)

    df_all = pd.concat([df_raw, ts_feats, adv_feats, opt_feats], axis=1)
    df_all = df_all.replace([np.inf,-np.inf], np.nan)

    # ── Cible ────────────────────────────────────────────────────────────────
    target, regime, _, _ = build_amplitude_target(vix_series, horizon, train_end_idx)
    df_all = df_all.reindex(target.index); df_all[TARGET_COL] = target

    # ── SHAP 2 passes ────────────────────────────────────────────────────────
    print(f"  [SHAP] ({time.time()-t_total:.1f}s)")
    feat_cols = [c for c in df_all.columns if c != TARGET_COL]
    df_tr  = df_all.loc[df_all.index <= train_end].dropna(subset=[TARGET_COL])
    X_base = df_tr[feat_cols].fillna(0).replace([np.inf,-np.inf],0)
    y_tr   = df_tr[TARGET_COL].values.astype(int)
    sc     = RobustScaler()
    X_sc   = pd.DataFrame(sc.fit_transform(X_base), columns=feat_cols, index=df_tr.index)
    top_base, _ = shap_select_features(X_sc, y_tr, CONFIG['top_n_shap'], 'base')
    idf         = generate_interactions(df_tr[top_base], top_base, top_n=20)
    df_ext      = pd.concat([df_tr[top_base],idf],axis=1).loc[:,~pd.concat([df_tr[top_base],idf],axis=1).columns.duplicated()]
    ext_cols    = df_ext.columns.tolist()
    sc2         = RobustScaler()
    X_ext       = pd.DataFrame(sc2.fit_transform(df_ext.fillna(0)), columns=ext_cols, index=df_tr.index)
    top_final, _ = shap_select_features(X_ext, y_tr, CONFIG['top_n_final'], 'final')
    print(f"  Features finales : {len(top_final)} ({time.time()-t_total:.1f}s)")

    # ── Préparer splits ───────────────────────────────────────────────────────
    def make_df_split(df_all, start, end, ext_cols, sc2, top_final):
        d = df_all.loc[(df_all.index > start) & (df_all.index <= end)].dropna(subset=[TARGET_COL])
        avail_top = [f for f in top_final if f in d.columns]
        inter_need = [f for f in top_final if f not in d.columns]
        idf2 = generate_interactions(d[avail_top], avail_top, top_n=len(avail_top)) if inter_need else pd.DataFrame(index=d.index)
        df_e = pd.concat([d[avail_top],idf2],axis=1)
        X_s  = pd.DataFrame(sc2.transform(df_e.reindex(columns=ext_cols).fillna(0)),
                              columns=ext_cols,index=d.index)[top_final].fillna(0)
        return X_s.values, d[TARGET_COL].values.astype(int), d.index, d

    X_val,  y_val,  val_idx,  df_val_raw  = make_df_split(df_all, train_end, val_end,   ext_cols, sc2, top_final)
    X_test, y_test, test_idx, df_test_raw = make_df_split(df_all, val_end,   all_dates[-1], ext_cols, sc2, top_final)
    X_tr_f = X_ext[top_final].fillna(0).values

    # ── Adversarial Validation ────────────────────────────────────────────────
    print(f"  [Adversarial Validation] ({time.time()-t_total:.1f}s)")
    adv_auc, drift_feats = adversarial_validation(X_tr_f, X_test, feature_names=top_final)

    # ── SMOTE ────────────────────────────────────────────────────────────────
    sampler_name, best_sampler = select_best_sampler(X_tr_f, y_tr)
    X_res, y_res = best_sampler.fit_resample(X_tr_f, y_tr)

    # ── DataLoaders ───────────────────────────────────────────────────────────
    def make_loader(X, y, shuffle=False, fit_sc=None):
        df_tmp = pd.DataFrame(X, columns=top_final); df_tmp[TARGET_COL] = y
        ds = VIXAmplitudeDataset(df_tmp, top_final, scaler=fit_sc)
        return DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=shuffle), ds.scaler

    df_res = pd.DataFrame(X_res, columns=top_final); df_res[TARGET_COL] = y_res
    vsp    = int(len(df_res)*0.85)
    sc_seq = RobustScaler().fit(X_res)
    ds_tr  = VIXAmplitudeDataset(df_res.iloc[:vsp], top_final, scaler=sc_seq)
    ds_va  = VIXAmplitudeDataset(df_res.iloc[vsp:], top_final, scaler=sc_seq)

    df_val_f  = pd.DataFrame(X_val,  columns=top_final); df_val_f[TARGET_COL]  = y_val
    df_test_f = pd.DataFrame(X_test, columns=top_final); df_test_f[TARGET_COL] = y_test
    ds_val_cal = VIXAmplitudeDataset(df_val_f,  top_final, scaler=sc_seq)
    ds_test    = VIXAmplitudeDataset(df_test_f, top_final, scaler=sc_seq)

    dl_tr   = DataLoader(ds_tr,      batch_size=CONFIG['batch_size'], shuffle=True)
    dl_va   = DataLoader(ds_va,      batch_size=256)
    dl_val_c= DataLoader(ds_val_cal, batch_size=256)
    dl_te   = DataLoader(ds_test,    batch_size=256)

    input_dim    = len(top_final)
    class_weights= compute_class_weights(y_res)

    # ── Entraînement 7 modèles ────────────────────────────────────────────────
    models_def = {
        'LSTM':        VIX_LSTM(input_dim),
        'TCN':         VIX_TCN(input_dim),
        'Transformer': VIX_Transformer(input_dim),
        'CNN-LSTM':    VIX_CNNLSTM(input_dim),
        'N-BEATS':     VIX_NBeats(input_dim, CONFIG['lookback']),
        'TFT':         VIX_TFT(input_dim),
        'Mamba':       VIX_Mamba(input_dim),
    }
    trained, base_res = {}, {}
    for name, model in models_def.items():
        print(f"\n  ── {name} ({time.time()-t_total:.1f}s) ──")
        model = model.to(device)
        train_model(model, dl_tr, dl_va, class_weights=class_weights, label=name)
        base_res[name] = evaluate_model(model, dl_te, label=name)
        trained[name]  = model

    # ── Temperature Scaling ───────────────────────────────────────────────────
    print(f"\n  [Temperature Scaling] ({time.time()-t_total:.1f}s)")
    calibrated = {}
    for name, model in trained.items():
        ts = TemperatureScaler(model).to(device)
        ts.calibrate(dl_val_c, device)
        calibrated[name] = ts

    # ── Stacking ─────────────────────────────────────────────────────────────
    print(f"\n  [Stacking] ({time.time()-t_total:.1f}s)")
    stacker    = MetaStackingClassifier(trained)
    stacker.fit(X_res, y_res, device)
    stack_pr   = stacker.predict_proba(X_test, device)
    dm         = {0:'DOWN',1:'DOWN',2:'UP',3:'UP'}
    stack_f1   = f1_score([dm[y] for y in y_test],[dm[p] for p in stack_pr.argmax(1)],average='macro',zero_division=0)
    print(f"  Stacking F1_dir = {stack_f1:.4f}")

    # ── Threshold Optimization ────────────────────────────────────────────────
    print(f"\n  [Threshold Opt.] ({time.time()-t_total:.1f}s)")
    thresholds = {}
    for name, cal in calibrated.items():
        xv = torch.tensor(X_val,dtype=torch.float32).unsqueeze(1).repeat(1,CONFIG['lookback'],1)
        pv = cal.predict_proba(xv)
        thr, f1t = optimize_direction_threshold(pv, y_val)
        thresholds[name] = thr
        print(f"    {name}: τ*={thr:.3f} F1={f1t:.4f}")

    # ── Conformal Prediction ──────────────────────────────────────────────────
    print(f"\n  [Conformal] ({time.time()-t_total:.1f}s)")
    best_name = max(base_res, key=lambda x: base_res[x]['F1_dir'])
    conformal = TemporalConformalClassifier(alpha=CONFIG['conformal_alpha'])
    conformal.calibrate(trained[best_name], dl_val_c, device)
    coverage, avg_sz = conformal.evaluate_coverage(trained[best_name], dl_te, device)

    # ── Ensemble Asymétrique ──────────────────────────────────────────────────
    print(f"\n  [Ensemble Asymétrique] ({time.time()-t_total:.1f}s)")
    vix_val_s  = vix_series.reindex(val_idx)
    vix_test_s = vix_series.reindex(test_idx)
    asym = AsymmetricEnsemble(trained)
    asym.fit_regime_weights(dl_val_c, vix_val_s, device)
    asym_f1 = asym.evaluate(dl_te, vix_test_s, device)

    # ── Stress Testing ────────────────────────────────────────────────────────
    print(f"\n  [Stress Testing] ({time.time()-t_total:.1f}s)")
    stress_res = stress_test_models(trained, df_all, top_final, TARGET_COL,
                                     sc_seq, val_end.strftime('%Y-%m-%d'))

    print(f"\n  Pipeline V2 terminé en {time.time()-t_total:.1f}s")
    return {
        'models': trained, 'calibrated': calibrated, 'stacker': stacker,
        'conformal': conformal, 'asym_ensemble': asym,
        'base_results': base_res, 'stacking_f1': stack_f1,
        'asymmetric_f1': asym_f1, 'conformal_coverage': coverage,
        'adv_auc': adv_auc, 'drift_features': drift_feats,
        'stress_results': stress_res, 'features': top_final,
        'thresholds': thresholds,
    }


## 17. Exécution

In [57]:
# Chargement déjà effectué (df_raw disponible depuis la cellule 2)
results_v2 = {}
for h in [5]:
    results_v2[h] = run_pipeline_v2(df_raw, horizon=h)

# ── Rapport ───────────────────────────────────────────────────────────────────
ML_REFS = {
    'LogReg N=9 (ML h=5j GLOBAL)':       {'F1_dir':0.634,'F1_UP_FORT':0.406,'F1_DOWN_FORT':0.575},
    'RandomForest N=8 (ML h=5j GLOBAL)': {'F1_dir':0.620,'F1_UP_FORT':0.412,'F1_DOWN_FORT':0.462},
}

print("\n" + "="*70)
print("RAPPORT FINAL — DL V2 vs Benchmarks ML")
print("="*70)
for h, res in results_v2.items():
    print(f"\n─── h={h}j ───")
    print(f"{'Modèle':<25} {'F1_dir':>8} {'F1_UP_FORT':>12} {'F1_DOWN_FORT':>14}")
    print("─"*62)
    for name, m in res['base_results'].items():
        print(f"  {name:<23} {m.get('F1_dir',0):>8.4f} {m.get('F1_UP_FORT',0):>12.4f} {m.get('F1_DOWN_FORT',0):>14.4f}")
    print(f"  {'STACKING':23} {res['stacking_f1']:>8.4f}")
    print(f"  {'ENSEMBLE ASYM.':23} {res['asymmetric_f1']:>8.4f}")
    print(f"  {'Conformal Coverage':23} {res['conformal_coverage']:>8.4f} (cible {1-CONFIG['conformal_alpha']:.2f})")
    print(f"  {'Adversarial AUC':23} {res['adv_auc']:>8.4f} ({'DRIFT' if res['adv_auc']>0.7 else 'OK'})")
    print("─"*62)
    for k,v in ML_REFS.items():
        print(f"  {k:<23} {v['F1_dir']:>8.4f} {v['F1_UP_FORT']:>12.4f} {v['F1_DOWN_FORT']:>14.4f}")

try:
    rows = []
    for h, res in results_v2.items():
        for name, m in res['base_results'].items():
            rows.append({'H':h,'Modèle':name,'Type':'DL Base',**m})
        rows.append({'H':h,'Modèle':'Stacking',   'Type':'DL Ensemble','F1_dir':res['stacking_f1']})
        rows.append({'H':h,'Modèle':'Asym Ens.',  'Type':'DL Ensemble','F1_dir':res['asymmetric_f1']})
    for k,v in ML_REFS.items():
        rows.append({'Modèle':k,'Type':'ML Benchmark',**v})
    pd.DataFrame(rows).to_excel('vix_dl_v2_report.xlsx', index=False, engine='xlsxwriter')
    print("\n[SAVE] vix_dl_v2_report.xlsx")
except Exception as e:
    print(f"[WARN] Export: {e}")



PIPELINE V2 | h=5j
  Train  → 2022-03-08 | Val → 2023-08-21 | Test → 2026-07-17
  [Features TS] (0.0s)
  EGARCH OK (0.2s)


  Kalman OK (40.1s)
  HMM OK (40.2s)
  Heston OK (41.2s)
  VRP OK (41.2s)
  Hawkes OK (43.3s)
  [Options Flow] (43.3s)
  [PCR] FRED non disponible (Unable to read URL: https://fred.stlouisfed.org/graph/fredgraph.csv?id=PUTCALLRATIO
Response Text:
b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n    <meta charset="utf-8">\r\n    <meta http-equiv="X-UA-Compatible" content="IE=edge">\r\n    <meta name="viewport" content="width=device-width, initial-scale=1">\r\n    <title>Error - St. Louis Fed</title>\r\n    <meta name="description" content="">\r\n    <meta name="keywords" content="">    \r\n    <link rel="stylesheet" type="text/css" href="/assets/bootstrap/dist/css/bootstrap.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/css/home.min.css?1553087253">\r\n    <link rel="stylesheet" type="text/css" href="/assets/fontawesome-free/css/all.min.css">\r\n    <link rel="stylesheet" type="text/css" href="/assets/select2/dist/css/select2.min.css">\r\n    <style>p {\r\n        marg

## 18. Résumé des modules V2

| Module | Principe | Gain attendu |
|---|---|---|
| **Temperature Scaling** | Paramètre $T$ sur logits, appris sur val | Probabilités calibrées → sizing fiable |
| **Stacking DL+ML** | XGBoost méta-modèle sur probs OOF | +2-5% F1_dir |
| **Threshold Opt.** | Seuil τ* optimal par régime sur val | +2-4% F1_dir sans re-training |
| **Options Flow** | PCR CBOE + corrélation implicite sectorielle | Signal amplitude UP_FORT |
| **Mamba (SSM)** | Convolutions causales O(n) | Long lookback sans mémoire quadratique |
| **Conformal Prediction** | Ensembles à 90% de couverture garantie | Sizing prudent sur les incertains |
| **Adversarial Validation** | AUC train vs test | Alerte drift de distribution |
| **Stress Testing** | 5 crises historiques séparées | Révèle les faiblesses cachées |
| **Ensemble Asymétrique** | Poids différents par régime VIX | Adaptation aux conditions courantes |
